# Two-stage Transfer Learning Task
In assignment 7A, a ChemBERTa model was fine-tuned to the ESOL dataset. Since that task took some considerable training time, the model was saved for further reuse, e.g. where only the regression head is retrained (which in contrast is a much cheaper operation).

Conceptually, this is a two-stage TL approach:

Foundation model ChemBERTa (general chemistry language) -> ESOL-tuned ChemBERTa (biased towards physicochemical descriptors) -> quick predictors (linear probes)

Reusing fine-tuned checkpoints as new model backbones is a routine operation to save computational time.

### Tasks
Note: The same random-state for splitting the dataset was used for all involved notebooks (`foundation_models.ipynb`, `7A_FineTuning.ipynb`).

1) Load the ESOL-tuned ChemBERTa model (encoder plus small regressor NN) and evaluate the predictions for the ESOL data (snippet provided)
2) In analogy to the notebook `foundation_models.ipynb` (session15/16), use the ESOL-tuned ChemBERTa model as fixed encoder and build a small machine learning model of your choice on top (e.g. ridge regression, RF, GB, ...)
3) Replace the dataset by the toxicity dataset (``tdc_ld50_zhu.csv``) and rerun the evaluation for the different transfer learning combinations (ChemBERTa+Regressor(retrain), ESOL-tuned ChemBERTa+Regressor(do not retrain), ESOL-tuned ChemBERTa+MLModel(retrain)), i.e. simply rerun the notebook with another dataset. Hint: you can crop the dataset size a bit by sampling so that retraining doesn't take too long (e.g. a GB model from task 2 took about 6 mins on my PC).
4) Complete the discussion points.


### Task 0: Import dependencies and data

In [14]:
from transformers import AutoModel, AutoTokenizer
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

Load the data including train test-split (use the same as in the other examples!)

In [2]:
df = pd.read_csv("esol.csv")
# df = pd.read_csv("tdc_ld50_zhu.csv")

# drop rows just in case either smiles or logS values are missing. 
# It is crucial to have complete and labelled data for our exercise!
df.dropna(axis=0, inplace=True)

print(f"Dataset size: {len(df)}")

Dataset size: 1128


In [3]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

Reuse Dataset class from assignment 7A.

In [4]:
class ESOLDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        smiles = self.df.iloc[idx]["smiles"]
        label = self.df.iloc[idx]["logS"]
        # label = self.df.iloc[idx]["ld_50"]

        enc = self.tokenizer(
            smiles,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.float)
        }


### Task 1:
Load and evaluate the ESOL-tuned ChemBERTa model.

In [5]:
# recreate model class
class chemberta_esol_regressor(nn.Module):
    
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.fc1 = nn.Linear(encoder.config.hidden_size, 256)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(256, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls = outputs.last_hidden_state[:, 0]
        x = self.act(self.fc1(cls))
        return self.fc2(x).squeeze(-1)

In [7]:
# load the pretrained encoder
encoder = AutoModel.from_pretrained("chemberta_esol_encoder")
tokenizer = AutoTokenizer.from_pretrained("chemberta_esol_encoder")

model = chemberta_esol_regressor(encoder)

# Load the pretrained weights for the regressor head:
head_state = torch.load("chemberta_esol_regressor_head.pt", map_location="cpu")

model.fc1.load_state_dict(head_state["fc1"])
model.fc2.load_state_dict(head_state["fc2"])

<All keys matched successfully>

Initialise the dataset and the loader.

In [8]:

test_dataset = ESOLDataset(val_df, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=64)


Evaluate the pretrained model:

In [9]:
# important: put model into evaluation mode (diables dropout and gradient)
model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for batch in test_loader:
        preds = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        y_true.append(batch["labels"].numpy())
        y_pred.append(preds.numpy())

y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print(f"Test RMSE: {rmse:.3f}")
print(f"Test MAE:  {mae:.3f}")
print(f"Test R²:   {r2:.3f}")

Test RMSE: 1.172
Test MAE:  0.917
Test R²:   0.709


### Task 2:
Use the ESOL-tuned ChemBERTa model as fixed encoder only and use its output as training input for a small ML model (not a NN) of your choice (= new trainable head). 

You define the encoder and tokenizer in analogy to the `foundation-models.ipynb`, likewise the smiles_encoding function, but you may have to change some small details.

Hint: Since the regression model is not a NN, you could use `return np.vstack(all_embeddings)` so that the embeddings are nicely compatible with any scikit-learn models.

In [10]:
# reuse the tuned ChemBERTa model as a fixed feature extractor
if "encoder" not in globals() or "tokenizer" not in globals():
    encoder = AutoModel.from_pretrained("chemberta_esol_encoder")
    tokenizer = AutoTokenizer.from_pretrained("chemberta_esol_encoder")

for param in encoder.parameters():
    param.requires_grad = False

encoder.eval()
target_col = "logS" if "logS" in train_df.columns else "ld_50"


In [11]:
def smiles_to_embeddings(smiles_list, encoder, tokenizer, batch_size=64, max_length=128):
    device = next(encoder.parameters()).device
    all_embeddings = []

    for start in range(0, len(smiles_list), batch_size):
        batch_smiles = smiles_list[start:start + batch_size]
        enc = tokenizer(
            batch_smiles,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt"
        )
        enc = {key: value.to(device) for key, value in enc.items()}

        with torch.no_grad():
            outputs = encoder(**enc)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()

        all_embeddings.append(cls_embeddings)

    return np.vstack(all_embeddings)


In [15]:
X_train = smiles_to_embeddings(train_df["smiles"].tolist(), encoder, tokenizer)
X_val = smiles_to_embeddings(val_df["smiles"].tolist(), encoder, tokenizer)

y_train = train_df[target_col].to_numpy()
y_val = val_df[target_col].to_numpy()

ridge_model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
ridge_model.fit(X_train, y_train)


Pipeline(steps=[('standardscaler', StandardScaler()), ('ridge', Ridge())])

In [16]:
train_pred = ridge_model.predict(X_train)
val_pred = ridge_model.predict(X_val)

train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
train_mae = mean_absolute_error(y_train, train_pred)
val_mae = mean_absolute_error(y_val, val_pred)
train_r2 = r2_score(y_train, train_pred)
val_r2 = r2_score(y_val, val_pred)

print(f"Train RMSE: {train_rmse:.3f}")
print(f"Val RMSE:   {val_rmse:.3f}")
print(f"Train MAE:  {train_mae:.3f}")
print(f"Val MAE:    {val_mae:.3f}")
print(f"Train R2:   {train_r2:.3f}")
print(f"Val R2:     {val_r2:.3f}")


Train RMSE: 0.416
Val RMSE:   1.295
Train MAE:  0.258
Val MAE:    0.963
Train R2:   0.960
Val R2:     0.645


### Task 3: Rerun with the toxicity dataset 
You can simply replace the imported dataframe. Note that depending on the model you chose in Task 2, training may take a bit - you can alleviate that problem by using a sample of the dataset.

You can run the General ChemBERTa + Regressor model in the original ``foundation_model.ipynb`` notebook (session 15/16).

In [17]:
#help

### Task 4: Discussion

1) Why is it important for comparing the generalisation/performance of the different models to have the same random-state for the train-test split considering the fine-tuning in 7A and the evaluation in 7B? What would the predictions tell you otherwise?
2) How did the performances of the three approaches compare for the ESOL dataset? Did the transfer learning stages improve the models?
3) Smaller models trained on molecular descriptors based on the smiles strings in the ESOL dataset (e.g. a GB model), delivered:
- Train RMSE: 0.386
- Test RMSE: 0.776
- Train R2: 0.965
- Test R2: 0.873
How do you judge that in comparison to the much more complicated models?
4) Discuss the results for using the three approaches on the toxicity dataset. Which one performed best? What is a clear no-go? Comment on target vs. source tasks in this context.
5) What would be a better approach for the toxicity?
6) How could we generally improve the performance?